# Rosette on Spider: zero-shot vs RAG few-shot vs fine-tuned NL→SQL

Three strategies for turning an English question into SQL, compared on the **Spider dev set**
(1034 questions, 20 databases unseen in Spider's training split):

| strategy | model | extra input |
|---|---|---|
| `zero_shot` | `google/flan-t5-small` | schema + question |
| `rag_fewshot` | `google/flan-t5-small` | + top-3 similar (question, SQL) pairs from Spider train, via TF-IDF |
| `fine_tuned` | `cssupport/t5-small-awesome-text-to-sql` | schema + question, in its training format |

Scored by **execution accuracy**: run the predicted and gold SQL, compare results
(ordered if the gold query has `ORDER BY`, as a multiset otherwise).

**Contamination:** ~55% of Spider dev appears verbatim in the fine-tuned checkpoint's training
data, so every number is reported on *seen*, *unseen* and *all*. Only *unseen* measures generalization.

**Kaggle:** Settings → *Accelerator: GPU T4* and *Internet: On*. Runs top to bottom in about 6 minutes on a T4.

In [ ]:
# ---- config ----
import os
REPO_URL = "https://github.com/ZephyrousChimes/rosette-spider.git"
LIMIT = int(os.environ["LIMIT"]) if os.environ.get("LIMIT") else None   # None = full dev set
RECOMPUTE_CONTAMINATION = False   # True re-streams ~650MB of training data to rebuild the seen-list
BATCH_SIZE = 32

In [ ]:
# ---- setup: find (or clone) the scripts ----
import subprocess, sys
from pathlib import Path

def find_root():
    for p in [Path.cwd(), Path.cwd() / "rosette-spider", Path("/kaggle/working/rosette-spider")]:
        if (p / "src" / "spider_data.py").exists():
            return p
    target = Path("/kaggle/working/rosette-spider") if Path("/kaggle").exists() else Path.cwd() / "rosette-spider"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
    return target

ROOT = find_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

try:
    import sentencepiece  # T5Tokenizer needs it
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=True)

In [ ]:
# ---- device ----
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print("gpu:", name, "compute capability", cap)
    if cap < (7, 0):
        print("WARNING: recent PyTorch builds may not support this GPU (P100 = 6.0). Switch the accelerator to T4.")
else:
    print("WARNING: no GPU. This will work but takes ~30+ minutes on CPU.")

## 1. Data: download Spider dev, verify it against the official release

In [ ]:
import spider_data
spider_data.download()
spider_data.verify()
dev = spider_data.load_dev()
train = spider_data.load_train()
print(f"dev: {len(dev)} questions over {len({x['db_id'] for x in dev})} databases | train (RAG bank): {len(train)}")

## 2. Contamination: which dev questions did the fine-tuned model train on?
The seen-list ships in `artifacts/spider_dev_seen.json`; set `RECOMPUTE_CONTAMINATION = True` to rebuild it from the raw training sets.

In [ ]:
import contamination
if RECOMPUTE_CONTAMINATION:
    contamination.compute()
seen = contamination.load()
print(f"{len(seen)}/{len(dev)} dev questions ({len(seen)/len(dev):.1%}) appear verbatim in the fine-tuned model's training data")

## 3. Prompts (one example of each)

In [ ]:
import strategies, metrics
if LIMIT:
    dev = dev[:LIMIT]
    print(f"LIMIT={LIMIT}: evaluating a subset")
retriever = strategies.Retriever(train)
prompts = strategies.build_prompts(dev, retriever)
for name, ps in prompts.items():
    print(f"===== {name} =====\n{ps[0]}\n")

## 4. Execute gold SQL, generate with each strategy, score

In [ ]:
gold = [metrics.run_sql(x["db_id"], x["query"]) for x in dev]
assert all(err is None for _, err in gold), "a gold query failed to execute"

records = [{"i": i, "db_id": x["db_id"], "question": x["question"], "gold_sql": x["query"],
            "seen_in_ft_training": i in seen, "structure": metrics.structure(x["query"])}
           for i, x in enumerate(dev)]

for name, ps in prompts.items():
    preds = strategies.run_or_load(name, ps, DEVICE, BATCH_SIZE)
    metrics.score(records, name, preds, gold)

## 5. Results

In [ ]:
import pandas as pd, report
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
acc = report.accuracy_table(records)
acc

In [ ]:
# paired exact McNemar: same questions, so compare the discordant pairs
report.paired_tests(records)

In [ ]:
# accuracy by gold-query structure
report.structure_table(records)

In [ ]:
# what kind of failure each strategy makes
report.failure_table(records)

## 6. Examples: where the fine-tuned model fails on unseen questions

In [ ]:
import random
random.seed(0)
fails = [r for r in records if not r["seen_in_ft_training"] and not r["fine_tuned"]["correct"]]
for r in random.sample(fails, min(8, len(fails))):
    print(f"[{r['db_id']}] {r['question']}")
    print(f"   gold: {r['gold_sql']}")
    print(f"   pred: {r['fine_tuned']['pred_sql']}")
    print(f"   kind: {report.failure_kind(r, 'fine_tuned')}\n")

## 7. Save

In [ ]:
import json
out = ROOT / "artifacts"
with open(out / "spider_records.jsonl", "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")
acc.to_csv(out / "accuracy.csv", index=False)
report.paired_tests(records).to_csv(out / "paired_tests.csv", index=False)
report.structure_table(records).to_csv(out / "by_structure.csv", index=False)
report.failure_table(records).to_csv(out / "failures.csv")
print("saved to", out)

## 8. Try it: ask a question, get SQL
`nl2sql.NL2SQL` wraps the fine-tuned model (the best of the three) for single queries. `src/serve.py`
serves the same function over HTTP. Give it a Spider database (`db_id`, and optionally `execute=True`),
or any `CREATE TABLE` schema text. Expect roughly 10% of answers to be right, per the table above.

In [ ]:
from nl2sql import NL2SQL
nl2sql = NL2SQL(DEVICE)
for q, db in [("How many singers do we have?", "concert_singer"),
              ("What is the average age of all singers?", "concert_singer"),
              ("List the names of all teachers.", "course_teach"),
              ("How many countries are there?", "car_1")]:
    r = nl2sql.ask(q, db_id=db, execute=True)
    print(f"[{db}] {q}\n   sql:  {r['sql']}\n   rows: {r['rows'][:5] if r['rows'] is not None else None}   error: {r['error']}\n")

In [ ]:
# any schema works for generation (no database to execute against)
shop = ("CREATE TABLE customers (customer_id INTEGER, name VARCHAR, city VARCHAR, signup_date VARCHAR); "
        "CREATE TABLE orders (order_id INTEGER, customer_id INTEGER, quantity INTEGER, order_date VARCHAR)")
for q in ["How many customers are there?", "What is the total quantity ordered by customer 1?"]:
    print(q, "->", nl2sql.ask(q, schema=shop)["sql"])